# PyTorch Tutorial 49: On-Device LLMs for Beginners

**Author:** PyTorch Tutorial Series  
**Date:** 2026  
**Prerequisites:** Notebooks 46-48 + basic LLM awareness (Notebooks 09-10 helpful)  
**Time:** ~1.5 hours

---

## What You'll Learn

1. **Why run LLMs on your phone/laptop** instead of the cloud
2. **Small Language Models (SLMs)** — the 2026 landscape
3. **INT4 quantization** — how it makes LLMs fit on phones
4. **KV cache** — why LLMs are memory-hungry and how to manage it
5. **Hands-on:** Build and run a tiny transformer locally and measure performance

---

## 1. LLMs on Your Phone?

ChatGPT, Claude, and Gemini run on **massive servers** — racks of GPUs with hundreds of gigabytes of memory. But what if you could run a language model **right on your phone**?

Think of it like this: a full orchestra (cloud LLM) can play any symphony, but a skilled solo guitarist (on-device SLM) can still play beautiful music — and they fit in your living room.

### Why Would You Want This?

| Benefit | Explanation |
|---------|-------------|
| **Privacy** | Your conversations never leave your device. No server ever sees your data. |
| **Offline** | Works on an airplane, in a tunnel, or anywhere without internet. |
| **No API costs** | Once deployed, every inference is free — no pay-per-token. |
| **Low latency** | No network round trip. Responses start immediately. |

### The Catch

Phones have **~4-8 GB RAM** (shared with the OS and other apps). A full Llama 3 70B model needs ~140 GB at FP16. That is **17x more than your phone has**.

So we need **much smaller models** and clever tricks to squeeze them in.

### Real Examples You Already Use

- **Predictive text** on your keyboard — a small language model suggests your next word
- **Gemini Nano** on Google Pixel phones — summarizes messages, suggests replies
- **Apple Intelligence** on iPhone — rewrites text, summarizes notifications
- **On-device voice assistants** — basic commands processed locally before hitting the cloud

## 2. Small Language Models (SLMs) — The 2026 Landscape

A **Small Language Model** is an LLM with fewer parameters — typically **135 million to 3.8 billion** (compared to 70B+ for the big ones).

Think of parameters like the neurons in a brain. Fewer neurons means less memory and less knowledge, but for many tasks (text classification, summarization, simple Q&A), a small brain is plenty.

### Key Models in 2026

| Model | Parameters | FP16 Size | Typical Use Case |
|-------|-----------|-----------|------------------|
| SmolLM2-135M | 135M | ~270 MB | Keyword extraction, simple classification |
| SmolLM2-1.7B | 1.7B | ~3.4 GB | On-device chat, summarization |
| Gemma 3 1B | 1B | ~2 GB | Mobile assistants, text rewriting |
| Llama 3.2 1B | 1.3B | ~2.6 GB | General-purpose on-device tasks |
| Phi-4-mini | 3.8B | ~7.6 GB | Best quality SLM, needs quantization for phones |

Let's calculate exactly **how much memory** these models need at different precisions.

In [ ]:
import torch
import torch.nn as nn
import math
import time

print("PyTorch version:", torch.__version__)
print("Device: CPU (this tutorial runs entirely on CPU)")
print()

In [ ]:
def calc_model_memory(num_params_billions, precision_bits):
    """Calculate memory needed for a model at a given precision."""
    bytes_per_param = precision_bits / 8
    size_gb = num_params_billions * bytes_per_param
    return size_gb


# Models and their parameter counts (in billions)
models_info = [
    ("SmolLM2-135M", 0.135),
    ("Gemma 3 1B", 1.0),
    ("Llama 3.2 1B", 1.3),
    ("SmolLM2-1.7B", 1.7),
    ("Phi-4-mini 3.8B", 3.8),
]

precisions = [("FP32", 32), ("FP16", 16), ("INT8", 8), ("INT4", 4)]

# Print header
header = f"{'Model':<20}"
for name, _ in precisions:
    header += f" {name:>8}"
print(header)
print("-" * 56)

# Print each model's memory at each precision
for model_name, params_b in models_info:
    row = f"{model_name:<20}"
    for _, bits in precisions:
        size = calc_model_memory(params_b, bits)
        row += f" {size:>6.1f}GB"
    print(row)

print()
print("A phone has ~4-8 GB RAM (shared with OS).")
print("This is why INT4 quantization is essential for on-device LLMs!")

## 3. Making LLMs Edge-Friendly

### INT4 Quantization: The Key Trick

Normally, each model weight is stored as a **32-bit float** (FP32) — that is 4 bytes per number.

**Quantization** means storing weights with fewer bits:
- FP32 (32 bits) -> FP16 (16 bits) = **2x smaller**
- FP16 (16 bits) -> INT8 (8 bits) = **2x smaller again**
- INT8 (8 bits) -> INT4 (4 bits) = **2x smaller again**

Total: FP32 to INT4 = **8x smaller!**

Think of it like JPEG compression for photos. The file gets much smaller, and you lose a tiny bit of quality — but for most uses, you cannot tell the difference.

### GGUF Format

**GGUF** is a file format designed specifically for running quantized LLMs on CPUs. It is the standard format used by `llama.cpp` and most local LLM tools. When you see a model file like `phi-4-mini-Q4_K_M.gguf`, the `Q4` tells you it uses 4-bit quantization.

### Let's See the Math

In [ ]:
# Demonstrate how quantization shrinks a weight tensor
num_weights = 1_000_000  # 1 million weights (a small layer)

print("Memory for 1 million weights at different precisions:")
print(f"  FP32 (32 bits): {num_weights * 4 / 1024 / 1024:.2f} MB")
print(f"  FP16 (16 bits): {num_weights * 2 / 1024 / 1024:.2f} MB")
print(f"  INT8  (8 bits): {num_weights * 1 / 1024 / 1024:.2f} MB")
print(f"  INT4  (4 bits): {num_weights * 0.5 / 1024 / 1024:.2f} MB")
print()

# Show what happens to actual values
original = torch.randn(8)  # 8 random FP32 weights
print("Original FP32 weights:")
print(f"  {original.tolist()}")
print()

# Simulate INT8 quantization (scale to -128..127 range)
scale = original.abs().max() / 127
quantized_int8 = torch.round(original / scale).clamp(-128, 127).to(torch.int8)
dequantized = quantized_int8.float() * scale

print("After INT8 quantization and dequantization:")
print(f"  {dequantized.tolist()}")
print(f"  Max error: {(original - dequantized).abs().max().item():.6f}")
print("  (Very small error — the model barely notices!)")

### Building a Tiny Transformer From Scratch

Instead of downloading a huge model, let's **build our own tiny transformer** to understand how LLMs work at the architecture level. This will be small enough to run on any CPU in seconds.

In [ ]:
class TinyAttention(nn.Module):
    """A single multi-head attention layer.

    Attention is how the model decides which earlier words
    are relevant when predicting the next word.
    """

    def __init__(self, hidden_dim, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        # These linear layers create the queries, keys, and values
        self.qkv = nn.Linear(hidden_dim, 3 * hidden_dim, bias=False)
        self.proj = nn.Linear(hidden_dim, hidden_dim, bias=False)

    def forward(self, x):
        batch, seq_len, hidden = x.shape
        # Split into queries, keys, values
        qkv = self.qkv(x).reshape(batch, seq_len, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)  # (3, batch, heads, seq, head_dim)
        q, k, v = qkv[0], qkv[1], qkv[2]
        # Scaled dot-product attention
        scale = math.sqrt(self.head_dim)
        scores = torch.matmul(q, k.transpose(-2, -1)) / scale
        # Causal mask: each token can only attend to previous tokens
        mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
        scores = scores.masked_fill(mask.to(x.device), float('-inf'))
        attn = torch.softmax(scores, dim=-1)
        out = torch.matmul(attn, v)
        # Reshape back and project
        out = out.transpose(1, 2).reshape(batch, seq_len, hidden)
        return self.proj(out)

In [ ]:
class TinyTransformerBlock(nn.Module):
    """One transformer block = attention + feed-forward network."""

    def __init__(self, hidden_dim, num_heads):
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.attn = TinyAttention(hidden_dim, num_heads)
        self.norm2 = nn.LayerNorm(hidden_dim)
        # Feed-forward: expand to 4x, then shrink back
        self.ffn = nn.Sequential(
            nn.Linear(hidden_dim, 4 * hidden_dim),
            nn.GELU(),
            nn.Linear(4 * hidden_dim, hidden_dim),
        )

    def forward(self, x):
        # Residual connections: add the output back to the input
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x

In [ ]:
class TinyLLM(nn.Module):
    """A minimal GPT-like language model.

    This is the same architecture as GPT-2, just much smaller.
    Real LLMs use this exact structure with more layers and bigger dimensions.
    """

    def __init__(self, vocab_size, hidden_dim, num_heads, num_layers, max_seq_len):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, hidden_dim)
        self.pos_emb = nn.Embedding(max_seq_len, hidden_dim)
        self.layers = nn.ModuleList(
            [TinyTransformerBlock(hidden_dim, num_heads) for _ in range(num_layers)]
        )
        self.norm = nn.LayerNorm(hidden_dim)
        self.head = nn.Linear(hidden_dim, vocab_size, bias=False)
        self.max_seq_len = max_seq_len

    def forward(self, token_ids):
        """Forward pass: token IDs in, next-token logits out."""
        batch, seq_len = token_ids.shape
        positions = torch.arange(seq_len, device=token_ids.device)
        x = self.token_emb(token_ids) + self.pos_emb(positions)
        for layer in self.layers:
            x = layer(x)
        x = self.norm(x)
        logits = self.head(x)  # (batch, seq_len, vocab_size)
        return logits

In [ ]:
# Create our tiny LLM
VOCAB_SIZE = 1000   # Real LLMs use 32k-128k
HIDDEN_DIM = 256    # Real LLMs use 2048-8192
NUM_HEADS = 4       # Real LLMs use 16-64
NUM_LAYERS = 4      # Real LLMs use 24-80
MAX_SEQ_LEN = 128   # Real LLMs use 2048-128k

model = TinyLLM(VOCAB_SIZE, HIDDEN_DIM, NUM_HEADS, NUM_LAYERS, MAX_SEQ_LEN)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
size_mb = total_params * 4 / (1024 * 1024)
print(f"Our tiny LLM:")
print(f"  Parameters:   {total_params:,}")
print(f"  Size (FP32):  {size_mb:.1f} MB")
print(f"  Size (FP16):  {size_mb / 2:.1f} MB")
print(f"  Size (INT8):  {size_mb / 4:.1f} MB")
print(f"  Size (INT4):  {size_mb / 8:.1f} MB")

## 4. KV Cache — The Memory Challenge

When an LLM generates text, it produces one token at a time. For each new token, it needs to "remember" everything it has already read. This memory is called the **KV cache** (Key-Value cache).

Think of it like reading a book and taking notes. For every new sentence you write, you look back at **all** your notes. The longer the text, the more notes pile up.

### Why It Matters for Phones

The KV cache grows with every token generated:

```
cache_size = 2 x num_layers x seq_len x hidden_dim x bytes_per_element
```

The `2` is because we store both **keys** and **values** at every layer.

For a 1B parameter model generating a 2048-token response, the KV cache alone can use **hundreds of megabytes**.

In [ ]:
def calc_kv_cache_size_mb(num_layers, seq_len, hidden_dim, bytes_per_elem=2):
    """Calculate KV cache memory in MB.

    The factor of 2 accounts for both keys and values.
    Default bytes_per_elem=2 assumes FP16 storage.
    """
    cache_bytes = 2 * num_layers * seq_len * hidden_dim * bytes_per_elem
    return cache_bytes / (1024 * 1024)


# KV cache for our tiny model
print("KV cache for our Tiny LLM (4 layers, 256 hidden dim):")
print(f"{'Sequence Length':>16} {'Cache (FP16)':>14} {'Cache (INT8)':>14}")
print("-" * 48)
for seq_len in [32, 64, 128, 256, 512]:
    fp16 = calc_kv_cache_size_mb(NUM_LAYERS, seq_len, HIDDEN_DIM, 2)
    int8 = calc_kv_cache_size_mb(NUM_LAYERS, seq_len, HIDDEN_DIM, 1)
    print(f"{seq_len:>16} {fp16:>12.2f} MB {int8:>12.2f} MB")

In [ ]:
# Now let's see KV cache for REAL models — this is where it gets scary
print("KV cache for real models (FP16, 2048 tokens):")
print(f"{'Model':<20} {'Layers':>8} {'Hidden':>8} {'Cache':>10}")
print("-" * 50)

real_models = [
    ("SmolLM2-135M", 12, 576),
    ("Gemma 3 1B", 26, 1152),
    ("Llama 3.2 1B", 16, 2048),
    ("Phi-4-mini 3.8B", 32, 3072),
    ("Llama 3 70B", 80, 8192),  # For comparison
]

for name, layers, hidden in real_models:
    cache = calc_kv_cache_size_mb(layers, 2048, hidden, 2)
    print(f"{name:<20} {layers:>8} {hidden:>8} {cache:>8.0f} MB")

print()
print("Strategies to manage KV cache on phones:")
print("  1. Limit context length (e.g., 512 instead of 2048)")
print("  2. Quantize cache to INT8 (halves cache memory)")
print("  3. Use grouped-query attention (GQA) to share keys/values")

## 5. Hands-On: Generate Text With Our Tiny Transformer

Our model is untrained (random weights), so the output will be gibberish — but the **process** is exactly how real LLMs generate text:

1. Feed in a sequence of token IDs
2. Model outputs a probability for each possible next token
3. Pick the most likely token (greedy decoding)
4. Append it to the sequence and repeat

In [ ]:
# Create a fake vocabulary for demonstration
# (Real LLMs use tokenizers like BPE with 32k+ tokens)
fake_vocab = [
    "the", "a", "is", "of", "and", "to", "in", "it", "that", "for",
    "on", "with", "as", "at", "by", "from", "are", "was", "be", "have",
    "this", "will", "not", "but", "they", "we", "can", "all", "or", "if",
    "model", "data", "phone", "device", "small", "fast", "run", "local",
    "memory", "token", "cache", "layer", "AI", "edge", "cloud", "query",
]
# Pad vocabulary to match VOCAB_SIZE
while len(fake_vocab) < VOCAB_SIZE:
    fake_vocab.append(f"word_{len(fake_vocab)}")

print(f"Vocabulary size: {len(fake_vocab)} tokens")
print(f"First 20 tokens: {fake_vocab[:20]}")

In [ ]:
@torch.no_grad()
def generate_greedy(target_model, start_tokens, max_new_tokens, vocab):
    """Generate text one token at a time using greedy decoding.

    Greedy = always pick the highest-probability next token.
    Returns generated text and performance metrics.
    """
    model_device = next(target_model.parameters()).device
    tokens = start_tokens.clone().to(model_device)
    generated_words = []
    times_per_token = []

    for step in range(max_new_tokens):
        # Truncate to max sequence length
        input_tokens = tokens[:, -target_model.max_seq_len:]
        start_time = time.perf_counter()
        logits = target_model(input_tokens)
        elapsed = time.perf_counter() - start_time
        times_per_token.append(elapsed)
        # Take logits for the last position only
        next_token_logits = logits[:, -1, :]
        next_token = next_token_logits.argmax(dim=-1, keepdim=True)
        tokens = torch.cat([tokens, next_token], dim=1)
        generated_words.append(vocab[next_token.item()])

    avg_time = sum(times_per_token) / len(times_per_token)
    tokens_per_sec = 1.0 / avg_time if avg_time > 0 else 0
    return generated_words, tokens_per_sec, times_per_token


# Generate some tokens
prompt = torch.tensor([[0, 30, 2, 34]])  # "the model is small"
prompt_words = [fake_vocab[i] for i in prompt[0].tolist()]
print(f"Prompt: {' '.join(prompt_words)}")
print()

In [ ]:
# Generate 20 tokens and measure performance
model.train(False)  # Set to inference mode
words, tok_per_sec, token_times = generate_greedy(
    model, prompt, max_new_tokens=20, vocab=fake_vocab
)

print(f"Generated: {' '.join(words)}")
print(f"(Random weights = gibberish — but the process is real!)")
print()
print(f"Performance:")
print(f"  Tokens per second: {tok_per_sec:.1f}")
print(f"  Avg time per token: {sum(token_times)/len(token_times)*1000:.1f} ms")
print()

# Show how generation time grows with sequence length
print("Time per token as sequence grows (no KV cache):")
for i, t in enumerate(token_times):
    bar = '#' * int(t * 5000)
    print(f"  Token {i+1:>2}: {t*1000:>6.1f} ms {bar}")
print()
print("Notice: each token takes longer because the model")
print("re-processes the ENTIRE growing sequence every time.")
print("KV caching fixes this by remembering previous computations.")

In [ ]:
# Measure peak memory usage during generation
import tracemalloc

tracemalloc.start()

# Run generation
words, tok_per_sec, _ = generate_greedy(
    model, prompt, max_new_tokens=50, vocab=fake_vocab
)

current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"Memory during generation of 50 tokens:")
print(f"  Current: {current / 1024 / 1024:.1f} MB")
print(f"  Peak:    {peak / 1024 / 1024:.1f} MB")
print(f"  Speed:   {tok_per_sec:.1f} tokens/sec")
print()
print("For reference, reading speed is ~4 tokens/sec.")
print("So even a small model on CPU can be fast enough!")

## 6. Quantizing Our Tiny LLM

Let's apply **dynamic quantization** to our tiny transformer and compare:
- Model size (FP32 vs INT8)
- Inference speed
- Output differences

Dynamic quantization converts the weight matrices to INT8 at load time and performs INT8 arithmetic during inference. It is the easiest form of quantization — one line of code.

In [ ]:
import os
import tempfile


def get_file_size_mb(model_to_save):
    """Save model to a temp file and measure its size on disk."""
    with tempfile.NamedTemporaryFile(delete=False, suffix='.pt') as f:
        torch.save(model_to_save.state_dict(), f.name)
        size = os.path.getsize(f.name) / (1024 * 1024)
        os.unlink(f.name)
    return size


# Original FP32 model
fp32_size = get_file_size_mb(model)
print(f"FP32 model size on disk: {fp32_size:.2f} MB")

In [ ]:
# Apply dynamic quantization (INT8)
quantized_model = torch.ao.quantization.quantize_dynamic(
    model,
    {nn.Linear},  # Quantize all Linear layers
    dtype=torch.qint8,
)

int8_size = get_file_size_mb(quantized_model)
print(f"INT8 model size on disk: {int8_size:.2f} MB")
print(f"Compression ratio: {fp32_size / int8_size:.1f}x smaller")

In [ ]:
# Compare speed: FP32 vs INT8
def benchmark_generation(mdl, label, num_tokens=30):
    """Benchmark token generation speed for a model."""
    mdl.train(False)
    _, tok_sec, times = generate_greedy(
        mdl, prompt, max_new_tokens=num_tokens, vocab=fake_vocab
    )
    avg_ms = sum(times) / len(times) * 1000
    print(f"  {label}: {tok_sec:.1f} tok/s, {avg_ms:.1f} ms/token")
    return tok_sec


print("Generation speed comparison:")
fp32_speed = benchmark_generation(model, "FP32")
int8_speed = benchmark_generation(quantized_model, "INT8")

In [ ]:
# Compare outputs (are they different?)
test_input = torch.tensor([[1, 2, 3, 4, 5]])

model.train(False)
quantized_model.train(False)

with torch.no_grad():
    fp32_out = model(test_input)
    int8_out = quantized_model(test_input)

# Check difference
diff = (fp32_out - int8_out).abs()
print("Output comparison (FP32 vs INT8):")
print(f"  Mean absolute difference: {diff.mean().item():.6f}")
print(f"  Max absolute difference:  {diff.max().item():.6f}")
print()

# Do they predict the same next token?
fp32_next = fp32_out[:, -1, :].argmax(dim=-1).item()
int8_next = int8_out[:, -1, :].argmax(dim=-1).item()
print(f"  FP32 predicts next token: {fp32_next} ({fake_vocab[fp32_next]})")
print(f"  INT8 predicts next token: {int8_next} ({fake_vocab[int8_next]})")
match = "YES" if fp32_next == int8_next else "NO"
print(f"  Same prediction? {match}")
print()

# Summary table
print("Summary:")
print(f"{'Metric':<25} {'FP32':>10} {'INT8':>10}")
print("-" * 47)
print(f"{'Size on disk':<25} {fp32_size:>8.2f}MB {int8_size:>8.2f}MB")
speedup = int8_speed / fp32_speed if fp32_speed > 0 else 0
print(f"{'Relative speed':<25} {'1.0x':>10} {f'{speedup:.1f}x':>10}")
print(f"{'Output quality':<25} {'Baseline':>10} {'~Same':>10}")

## 7. The Deployment Stack in 2026

Once you have a small, quantized model, you need a **runtime** to actually run it on a device. Here are the main options:

| Framework | By | Best For | Quantization | Platforms |
|-----------|-----|----------|-------------|----------|
| **ExecuTorch** | Meta/PyTorch | Llama models on mobile | INT4/INT8 | iOS, Android, microcontrollers |
| **llama.cpp / GGUF** | Community | Any GGUF model on CPU | Q2-Q8 variants | Mac, Linux, Windows, Android |
| **MediaPipe LLM** | Google | Gemma/Gemini Nano | INT4/INT8 | Android, iOS, Web |
| **CoreML** | Apple | Apple devices only | INT4/INT8/FP16 | iOS, macOS, watchOS |
| **ONNX Runtime** | Microsoft | Cross-platform | INT4/INT8 | Everywhere |

### Which Should You Pick?

- **Building a PyTorch model for mobile?** Use **ExecuTorch** — it is PyTorch's official mobile path.
- **Want to run an existing LLM locally on your laptop?** Use **llama.cpp** with a GGUF model file.
- **Building an Android/iOS app with Gemma?** Use **MediaPipe LLM Inference API**.
- **Apple ecosystem only?** **CoreML** gives the best performance on Apple hardware.
- **Need to run everywhere?** **ONNX Runtime** has the widest platform support.

## 8. When to Use On-Device vs Cloud

Not every task needs a cloud LLM, and not every task can run on a phone. Here is a simple decision tree:

```
Does the task need GPT-4 / Claude-level reasoning?
  YES -> Use cloud
  NO  -> Continue...

Is privacy critical (medical, financial, personal data)?
  YES -> On-device
  NO  -> Continue...

Must it work offline?
  YES -> On-device
  NO  -> Continue...

Is it a simple task (classification, extraction, rewriting)?
  YES -> On-device SLM (saves money!)
  NO  -> Continue...

Complex multi-step reasoning or long documents?
  YES -> Cloud (or hybrid: SLM for easy parts, cloud for hard parts)
  NO  -> On-device is probably fine
```

### The Hybrid Approach (Best of Both Worlds)

Many production apps in 2026 use a **hybrid** strategy:

1. **On-device SLM** handles simple, frequent tasks (autocomplete, classification, quick replies)
2. **Cloud LLM** handles complex tasks (long summarization, coding, creative writing)
3. The app **routes** each request to the right model based on complexity

This saves money (most requests are simple) while keeping quality high (complex tasks go to the cloud).

## 9. Try It Yourself

**Exercise 1: More Layers**  
Modify the `TinyLLM` to use 8 layers instead of 4. How does the generation speed change? How much bigger is the model?

```python
bigger_model = TinyLLM(VOCAB_SIZE, HIDDEN_DIM, NUM_HEADS, num_layers=8, max_seq_len=MAX_SEQ_LEN)
# Count parameters and benchmark it!
```

**Exercise 2: What Fits in 4GB?**  
Using the `calc_model_memory` function, calculate: what is the largest model (in billions of parameters) that fits in 4 GB of RAM at INT4 precision? Remember to leave ~1 GB for KV cache and OS overhead.

```python
# Hint: at INT4, each param = 0.5 bytes
# Available RAM = 4GB - 1GB (overhead) = 3GB
# max_params = 3GB / 0.5 bytes = ?
```

**Exercise 3: Token Limit**  
The `generate_greedy` function already has `max_new_tokens`. Try generating 100 tokens and observe how much slower the later tokens are compared to the first ones. Can you explain why?

## 10. Recap

**What we learned:**

- **On-device LLMs** let you run language models directly on phones and laptops — no internet, no API costs, full privacy
- **Small Language Models (SLMs)** with 135M to 3.8B parameters are designed for edge devices
- **INT4 quantization** shrinks models by 8x (FP32 to INT4), making a 3.8B model fit in ~2 GB
- **KV cache** grows with every generated token and can use hundreds of MB — limit context length or quantize the cache
- We **built a tiny transformer from scratch** and saw exactly how text generation works
- **Dynamic quantization** (INT8) reduces model size with minimal quality loss
- The 2026 deployment stack includes ExecuTorch, llama.cpp, MediaPipe, CoreML, and ONNX Runtime
- **Hybrid approaches** (on-device for simple tasks, cloud for complex ones) are the practical sweet spot

**Key takeaway:** You do not need a massive GPU to run a useful language model. With the right model size and quantization, your phone can handle many LLM tasks locally.

---

**What's next:** Notebook 50 covers **Federated Learning** — how to train models across many devices without ever sharing private data. This pairs perfectly with on-device inference: your phone runs the model AND helps improve it, all while keeping your data local.